# **Phishing URL Detection ? Dataset PhiUSIIL + Dataset Baru**

Notebook ini memuat alur analisis data, pelatihan, evaluasi, dan penyimpanan model deteksi URL phishing menggunakan dataset gabungan.

Tujuan: mempertahankan semua kolom asli pada CSV master (98 kolom), tetapi memakai **15 fitur URL yang konsisten** saat melatih dan menjalankan model di website. URL baru cukup dimasukkan sebagai teks; website tidak perlu membaca HTML atau melakukan DNS lookup.

**0 = Phishing, 1 = Legitimate.**

Data dibagi berdasarkan **registered domain** menjadi train/validation/test. Pemilihan model menggunakan validation, sedangkan evaluasi akhir menggunakan test. Analisis mencakup EDA, tiga algoritma ML, tuning opsional, confusion matrix, dan prediksi URL.

Modul `feature_extraction.py` harus berada di folder yang sama dengan notebook.


## 0. Persiapan VS Code

Buka folder proyek di VS Code, pilih interpreter/kernel Python yang benar, dan instal dependensi **melalui terminal VS Code**:

```bash
python -m pip install -r requirements.txt
```

Letakkan file berikut di **folder proyek yang sama**: `v2.ipynb`, `dataset_master_semua_fitur.csv`, `feature_extraction.py`, `requirements.txt`. Pastikan VS Code menjalankan notebook dengan working directory folder proyek.

Untuk uji coba cepat tanpa melatih semua baris, aktifkan `DEBUG_ROWS` di cell konfigurasi. **Untuk angka laporan akhir, DEBUG_ROWS harus 0 dan semua cell dijalankan ulang dari awal.**

### **1. Data Understanding**

#### **1.1 Import Library dan Loading Dataset**

In [ ]:
from pathlib import Path
import os, json, shutil, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay)
from feature_extraction import FEATURE_COLUMNS, normalize_url, extract_features, get_hostname

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
DEBUG_ROWS = int(os.getenv('PHISHING_DEBUG_ROWS', '0'))  # 0 = seluruh data; angka positif = sampel untuk tes
pd.set_option('display.max_columns', 30)
print('Versi scikit-learn:', sklearn.__version__)
print('Working directory:', Path.cwd())
print('Jumlah fitur URL untuk website:', len(FEATURE_COLUMNS))

In [ ]:
# CSV master tetap memiliki seluruh 98 kolom asli; hanya kolom relevan yang dibaca
# ke RAM untuk training agar VS Code tidak boros memori.
DATA_PATH = Path('dataset_master_semua_fitur.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'File tidak ditemukan: {DATA_PATH.resolve()}')

master_columns = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
phi_columns = [c for c in master_columns if c.startswith('phi_')]
baru_columns = [c for c in master_columns if c.startswith('baru_')]
dns_columns = [c for c in master_columns if c.startswith('dns_')]
model_columns = ['model_' + c for c in FEATURE_COLUMNS]
needed_columns = ['URL', 'label', 'source', 'feature_valid'] + model_columns
missing = [c for c in needed_columns if c not in master_columns]
if missing:
    raise ValueError(f'Kolom wajib tidak ditemukan di master: {missing}')

df = pd.read_csv(DATA_PATH, usecols=needed_columns,
                 dtype={'URL': 'string'}, low_memory=False)
df = df.rename(columns={'model_' + c: c for c in FEATURE_COLUMNS})
if DEBUG_ROWS > 0:
    df = df.sample(n=min(DEBUG_ROWS, len(df)), random_state=RANDOM_STATE).reset_index(drop=True)
    print('MODE DEBUG: hanya sampel untuk pengujian, BUKAN hasil laporan final')

print('Master lengkap:', len(master_columns), 'kolom')
print('Fitur PhiUSIIL:', len(phi_columns), '| fitur baru:', len(baru_columns),
      '| DNS:', len(dns_columns), '| model:*', len(model_columns))
print('Baris yang sedang dianalisis:', f'{len(df):,}')
print('\nDistribusi sumber:')
print(df['source'].value_counts())
df.head()

#### **1.2 Dataset Information**

In [ ]:
print('Informasi data terpilih untuk training:')
df.info()
print('\nJumlah nilai kosong per kolom:')
print(df.isna().sum().sort_values(ascending=False).head(20))
print('\nDistribusi label (sebelum cleaning):')
print(df['label'].value_counts(dropna=False))
print('\nDuplikat URL mentah:', df['URL'].duplicated().sum())

#### **1.3 Data Cleaning, Konflik Label, dan Duplicate Handling**

Penghapusan duplikat dilakukan **setelah** normalisasi URL, dengan label bertentangan disisihkan. File master asli **tidak dihapus/ditimpa**. Kolom kosong pada fitur `phi_*`, `baru_*`, `dns_*` tetap ada pada file master dan **tidak** otomatis diisi nol.


In [ ]:
def safe_normalize(value):
    try:
        return normalize_url(value)
    except (TypeError, ValueError):
        return pd.NA

before_clean = len(df)
df_work = df.dropna(subset=['URL', 'label']).copy()
df_work['label'] = pd.to_numeric(df_work['label'], errors='coerce')
df_work = df_work[df_work['label'].isin([0, 1])].copy()
df_work['label'] = df_work['label'].astype('int8')
df_work = df_work[df_work['feature_valid'].eq(1)].copy()
df_work['normalized_url'] = df_work['URL'].map(safe_normalize)
invalid_count = int(df_work['normalized_url'].isna().sum())
df_work = df_work.dropna(subset=['normalized_url'])

counts = df_work.groupby('normalized_url')['label'].nunique()
conflict_urls = set(counts[counts > 1].index)
conflict_rows = int(df_work['normalized_url'].isin(conflict_urls).sum())
df_clean = df_work.loc[~df_work['normalized_url'].isin(conflict_urls)].copy()
duplicates_removed = int(df_clean.duplicated('normalized_url').sum())
df_clean = df_clean.drop_duplicates('normalized_url').reset_index(drop=True)
df_clean['URL'] = df_clean['normalized_url']
df_clean['has_www'] = df_clean['URL'].map(lambda u: get_hostname(u).startswith('www.'))
assert df_clean['normalized_url'].is_unique
assert set(df_clean['label'].unique()) == {0, 1}

print('Sebelum cleaning:', f'{before_clean:,}')
print('URL invalid:', invalid_count)
print('URL dengan label konflik:', len(conflict_urls), '(baris:', conflict_rows, ')')
print('Duplikat normalisasi dibuang:', duplicates_removed)
print('URL setelah cleaning:', f'{len(df_clean):,}')
print('Kelas 0=Phishing, 1=Legitimate:')
print(df_clean['label'].value_counts())
print('Persentase URL TANPA www: {:.2%}'.format((~df_clean['has_www']).mean()))

## **2. Exploratory Data Analysis (EDA) — Data Gabungan**

#### **2.1 Distribusi Kelas dan Sumber Data**

In [ ]:
class_counts = df_clean['label'].value_counts().reindex([0, 1], fill_value=0)
print('Distribusi kelas:')
print(class_counts.rename(index={0: 'Phishing', 1: 'Legitimate'}))
print('\nDistribusi sumber:')
print(df_clean['source'].value_counts())
print('\nProporsi www:')
print(df_clean['has_www'].value_counts().rename(index={False:'Tanpa www', True:'Dengan www'}))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['Phishing', 'Legitimate'], class_counts.values)
ax.set_title('Distribusi Kelas — Dataset Gabungan')
ax.set_ylabel('Jumlah URL')
plt.tight_layout(); plt.show()

In [ ]:
# Audit per sumber dan penggunaan www, untuk laporan penelitian.
source_www = pd.crosstab(df_clean['source'], df_clean['has_www'])
source_www = source_www.rename(columns={False:'Tanpa www', True:'Dengan www'})
print(source_www)
print('\n% URL tanpa www pada setiap sumber:')
print((source_www['Tanpa www'] / source_www.sum(axis=1) * 100).round(2))
source_www.plot(kind='bar', stacked=True, figsize=(9, 5))
plt.title('Distribusi URL dengan/tanpa www menurut sumber')
plt.ylabel('Jumlah URL'); plt.xticks(rotation=15, ha='right')
plt.tight_layout(); plt.show()

#### **2.2 Karakteristik 15 Fitur URL**


In [ ]:
url_features = FEATURE_COLUMNS  # Sama dengan 15 fitur yang akan dipakai di backend website
print('Fitur:', url_features)
print(df_clean[url_features].describe().T.round(3))
corr = df_clean[url_features + ['label']].corr(numeric_only=True)['label'].drop('label').sort_values()
print('\nKorelasi fitur terhadap label 1=legitimate (negatif -> terkait phishing):')
print(corr.round(3))
fig, ax = plt.subplots(figsize=(9, 6))
corr.plot.barh(ax=ax)
ax.axvline(0, color='gray', linewidth=1)
ax.set_title('Korelasi Fitur URL terhadap Label (0=Phishing, 1=Legitimate)')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df_clean.boxplot(column='URLLength', by='label', ax=ax)
ax.set_title('Distribusi URL Length menurut Label')
ax.set_xlabel('Label (0=Phishing, 1=Legitimate)'); ax.set_ylabel('URL Length')
fig.suptitle('')
plt.tight_layout(); plt.show()

## **3. Feature Engineering untuk Website (15 Fitur)**

**PENTING:** 98 kolom tetap dipertahankan dalam `dataset_master_semua_fitur.csv` sebagai arsip penelitian. **15 kolom `model_*`** dihitung dengan definisi yang sama untuk seluruh URL pada dua sumber. Di dalam notebook namanya diubah menjadi nama biasa (misal `URLLength`). Fungsi `extract_features()` pada `feature_extraction.py` harus identik saat training dan saat website memprediksi URL baru.

In [ ]:
x_final = df_clean[FEATURE_COLUMNS].apply(pd.to_numeric, errors='raise').reset_index(drop=True)
y_final = df_clean['label'].reset_index(drop=True)
assert len(x_final) == len(y_final) == len(df_clean)
assert x_final.notna().all().all(), 'Ada nilai kosong dalam fitur model'
assert np.isfinite(x_final.to_numpy(dtype=float)).all(), 'Ada fitur tak hingga / NaN'
print('X:', x_final.shape, '| y:', y_final.shape)

# Konsistensi sampel dengan kode yang nanti digunakan oleh website
sample = df_clean.sample(n=min(100, len(df_clean)), random_state=RANDOM_STATE)
for _, row in sample.iterrows():
    expected = extract_features(row['URL'])
    for name in FEATURE_COLUMNS:
        if not np.isclose(row[name], expected[name], rtol=1e-9, atol=1e-9):
            raise AssertionError(f'Fitur {name} tidak konsisten untuk {row["URL"]}')
print('Tes konsistensi fitur pada sampel: OK')

In [ ]:
# File training ini opsional. Jangan mengira seluruh fitur asli di master telah hilang.
EXPORT_TRAIN_CSV = False
if EXPORT_TRAIN_CSV:
    training_export = pd.concat(
        [df_clean[['URL','label','source','has_www']].reset_index(drop=True), x_final], axis=1
    )
    training_export.to_csv('dataset_training_15_fitur.csv', index=False)
    print('Tersimpan: dataset_training_15_fitur.csv')

## **4. Domain-Based Train / Validation / Test Split**

Data dibagi berdasarkan **registered domain**. Domain yang sama (mis. `www.contoh.com` dan `login.contoh.com`) tidak boleh tersebar ke set yang berbeda. Gunakan `tldextract` agar registered domain seperti `contoh.co.id` diproses lebih baik; pustaka ini wajib diinstal lewat `requirements.txt`.

Proporsi 20%/25% pada pembagian kelompok tidak menjamin tepat 20% baris karena ukuran tiap kelompok domain bisa berbeda. **Pembagian harus dijalankan sebelum training dan tuning.**


In [ ]:
import tldextract
extract_tld = tldextract.TLDExtract(suffix_list_urls=None)  # Tidak mengunduh daftar saat runtime

def domain_group(url):
    host = get_hostname(url)
    ext = extract_tld(host)
    return ext.top_domain_under_public_suffix or host

groups = df_clean['URL'].map(domain_group).reset_index(drop=True)
print('Jumlah registered domains:', groups.nunique())

# Sisihkan TEST 20% berdasarkan domain, tidak digunakan untuk memilih model.
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
idx_trainval, idx_test = next(gss_test.split(x_final, y_final, groups=groups))

# Sisihkan VALIDATION dari trainval, tanpa mencampurkan kelompok domain.
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE + 1)
i_train, i_val = next(gss_val.split(
    x_final.iloc[idx_trainval], y_final.iloc[idx_trainval], groups=groups.iloc[idx_trainval]
))
idx_train, idx_val = idx_trainval[i_train], idx_trainval[i_val]

assert set(groups.iloc[idx_train]).isdisjoint(set(groups.iloc[idx_val]))
assert set(groups.iloc[idx_train]).isdisjoint(set(groups.iloc[idx_test]))
assert set(groups.iloc[idx_val]).isdisjoint(set(groups.iloc[idx_test]))

x_train, y_train = x_final.iloc[idx_train], y_final.iloc[idx_train]
x_val, y_val = x_final.iloc[idx_val], y_final.iloc[idx_val]
x_test, y_test = x_final.iloc[idx_test], y_final.iloc[idx_test]
for title, y in [('Train',y_train), ('Validation',y_val), ('Test',y_test)]:
    print(f'{title:<12} {len(y):>8,} baris, phishing={(y==0).sum():>8,}, legitimate={(y==1).sum():>8,}')
    assert set(y.unique()) == {0, 1}, f'Kelas hilang pada split {title}'
print('Overlap domain antarsplit: 0')

## **5. Model Training dan Evaluasi VALIDATION**

Pelatihan membandingkan Logistic Regression, Random Forest, dan HistGradientBoosting. **Phishing = label 0**, sehingga metrik phishing harus menggunakan `pos_label=0` dan ROC-AUC harus memakai probabilitas kelas 0; jika langsung mengambil `[:, 1]`, yang diukur adalah probabilitas legitimate.


In [ ]:
def model_metrics(estimator, X, y):
    pred = estimator.predict(X)
    phishing_idx = list(estimator.classes_).index(0)
    prob_phishing = estimator.predict_proba(X)[:, phishing_idx]
    return {
        'Accuracy': accuracy_score(y, pred),
        'Phishing Precision': precision_score(y, pred, pos_label=0, zero_division=0),
        'Phishing Recall': recall_score(y, pred, pos_label=0, zero_division=0),
        'Phishing F1': f1_score(y, pred, pos_label=0, zero_division=0),
        'ROC-AUC Phishing': roc_auc_score((np.asarray(y) == 0).astype(int), prob_phishing),
    }

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, random_state=RANDOM_STATE),
}
validation_rows = []
for name, model in models.items():
    print('Training:', name)
    model.fit(x_train, y_train)
    validation_rows.append({'Model': name, **model_metrics(model, x_val, y_val)})
comparison_val = (pd.DataFrame(validation_rows)
                  .sort_values(['Phishing F1', 'Phishing Recall', 'ROC-AUC Phishing'], ascending=False)
                  .reset_index(drop=True))
print('Perbandingan pada VALIDATION (bukan test):')
display(comparison_val.round(4))

In [ ]:
# Plot model comparison: memakai nilai validation yang benar-benar dihitung.
ax = comparison_val.set_index('Model')[['Phishing Recall', 'Phishing F1', 'ROC-AUC Phishing']].plot(
    kind='bar', figsize=(9, 5), rot=10)
ax.set_ylim(0, 1.03)
ax.set_title('Perbandingan Model — Validation (Domain-Based)')
ax.set_ylabel('Skor')
plt.tight_layout(); plt.show()

### **5.1 Random Forest Hyperparameter Tuning (Opsional)**

In [ ]:
# Atur True jika ingin tuning. Bisa lambat untuk 325 ribu URL.
RUN_RF_TUNING = False
if RUN_RF_TUNING:
    from sklearn.model_selection import RandomizedSearchCV, GroupKFold
    from sklearn.metrics import make_scorer
    search = RandomizedSearchCV(
        estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        param_distributions={
            'n_estimators': [100, 150, 200], 'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2'],
        },
        n_iter=8, cv=GroupKFold(n_splits=3),
        scoring=make_scorer(f1_score, pos_label=0, zero_division=0),
        random_state=RANDOM_STATE, n_jobs=-1, verbose=1
    )
    search.fit(x_train, y_train, groups=groups.iloc[idx_train])
    print('Parameter RF terbaik:', search.best_params_)
    models['Random Forest (Tuned)'] = search.best_estimator_
    validation_rows.append({'Model': 'Random Forest (Tuned)',
                            **model_metrics(search.best_estimator_, x_val, y_val)})
    comparison_val = (pd.DataFrame(validation_rows)
                      .sort_values(['Phishing F1', 'Phishing Recall', 'ROC-AUC Phishing'], ascending=False)
                      .reset_index(drop=True))
    display(comparison_val.round(4))
else:
    print('Tuning dilewati. Aktifkan RUN_RF_TUNING bila diperlukan.')

## **6. Final Model Evaluation pada TEST Domain Baru**

Model terbaik dipilih dari **validation**. Setelah dipilih, model dilatih ulang pada train + validation, dan **baru sekali** dievaluasi pada test yang disimpan sejak awal. Angka untuk BAB IV harus berasal dari output evaluasi yang dijalankan pada notebook ini.


In [ ]:
BEST_NAME = comparison_val.iloc[0]['Model']
final_model = clone(models[BEST_NAME])
final_model.fit(x_final.iloc[idx_trainval], y_final.iloc[idx_trainval])
final_pred = final_model.predict(x_test)
phish_idx = list(final_model.classes_).index(0)
final_phish_prob = final_model.predict_proba(x_test)[:, phish_idx]
metrics_test = model_metrics(final_model, x_test, y_test)
print('MODEL TERPILIH:', BEST_NAME)
print('\nHasil TEST:')
for k, v in metrics_test.items():
    print(f'{k:<20}: {v:.4f}')
print('\nClassification report (0=phishing, 1=legitimate):')
print(classification_report(y_test, final_pred, labels=[0, 1],
                            target_names=['Phishing', 'Legitimate'], zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_test, final_pred, labels=[0, 1],
    display_labels=['Phishing','Legitimate'], cmap='Blues', values_format='d')
plt.title(f'Confusion Matrix — Test ({BEST_NAME})')
plt.tight_layout(); plt.show()

### **6.1 Audit Hasil: Dengan/Tanpa `www` dan Sumber Dataset**

In [ ]:
test_audit = df_clean.iloc[idx_test][['URL', 'label', 'source', 'has_www']].copy().reset_index(drop=True)
test_audit['pred'] = final_pred
test_audit['phishing_score'] = final_phish_prob

def score_per_group(part):
    if not len(part):
        return pd.Series(dtype=float)
    return pd.Series({
        'Jumlah URL': len(part),
        'Phishing': int((part.label == 0).sum()),
        'Legitimate': int((part.label == 1).sum()),
        'Accuracy': accuracy_score(part.label, part.pred),
        'Phishing Recall': recall_score(part.label, part.pred, pos_label=0, zero_division=0),
        'Phishing F1': f1_score(part.label, part.pred, pos_label=0, zero_division=0),
    })

summary_www = pd.DataFrame({v: score_per_group(p) for v,p in test_audit.groupby('has_www')}).T
summary_www.index = summary_www.index.map({False:'Tanpa www', True:'Dengan www'})
summary_source = pd.DataFrame({v: score_per_group(p) for v,p in test_audit.groupby('source')}).T
print('Skor pada TEST menurut www:')
display(summary_www.round(4))
print('Skor pada TEST menurut sumber dataset:')
display(summary_source.round(4))
print('Contoh phishing yang salah diklasifikasi legitimate:')
display(test_audit.loc[(test_audit.label == 0) & (test_audit.pred == 1),
                       ['URL','source','phishing_score']].head(10))

In [ ]:
# Untuk tabel BAB IV — hanya simpan metrik yang sudah benar-benar dihitung.
results_dir = Path('results')
results_dir.mkdir(exist_ok=True)
comparison_val.to_csv(results_dir / 'hasil_validasi_model.csv', index=False)
pd.DataFrame([{'Model': BEST_NAME, **metrics_test}]).to_csv(
    results_dir / 'hasil_test_final.csv', index=False)
summary_www.to_csv(results_dir / 'hasil_test_www.csv')
summary_source.to_csv(results_dir / 'hasil_test_per_sumber.csv')
print('Tabel hasil tersimpan di folder:', results_dir.resolve())

## **7. Model Saving and Inference Testing**

Simpan **model final** beserta urutan 15 fitur, pemetaan label, nama model terpilih, metrik test, dan versi scikit-learn dalam satu bundle untuk digunakan saat inferensi.


In [ ]:
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)
bundle = {
    'model': final_model,
    'feature_columns': list(FEATURE_COLUMNS),
    'labels': {0:'Phishing', 1:'Legitimate'},
    'selected_model': BEST_NAME,
    'test_metrics': metrics_test,
    'trained_on': 'PhiUSIIL + dataset baru 2026; URL unik; domain-based split',
    'sklearn_version': sklearn.__version__,
}
MODEL_PATH = MODEL_DIR / 'phishing_url_bundle.joblib'
joblib.dump(bundle, MODEL_PATH, compress=3)
shutil.copy2('feature_extraction.py', MODEL_DIR / 'feature_extraction.py')
print('Model berhasil disimpan:', MODEL_PATH.resolve())

In [ ]:
artifact = joblib.load(MODEL_PATH)

def predict_url(url):
    model = artifact['model']
    input_features = pd.DataFrame([extract_features(url)], columns=artifact['feature_columns'])
    label = int(model.predict(input_features)[0])
    prob = model.predict_proba(input_features)[0]
    phish_index = list(model.classes_).index(0)
    return {
        'url': normalize_url(url),
        'label': label,
        'prediction': artifact['labels'][label],
        'phishing_score': round(float(prob[phish_index]), 4),
        'note': 'Klasifikasi berbasis pola URL; bukan jaminan keamanan situs.'
    }

# Uji alur URL langsung seperti pada website
for url in [
    'https://example.com/login',
    'https://www.example.com/login',
    'https://login.example.com/verify?id=1',
    'http://192.168.1.1/login',
]:
    print(predict_url(url))

### **7.1 File untuk backend website**

File `app_flask.py` sudah disediakan di folder proyek. Setelah model jadi, copy file itu bersama `feature_extraction.py` dan `models/phishing_url_bundle.joblib` ke folder server website. Tidak perlu membawa CSV ke server.

Jalankan Flask dari terminal VS Code:

```bash
python app_flask.py
```

Kirim `POST http://127.0.0.1:5000/predict` dengan JSON `{"url": "https://example.com/login"}`. Prediksi hanya menganalisis teks URL, tidak mengunjungi alamat tersebut.

In [ ]:
# Buat ZIP yang siap dipindahkan ke server website (setelah training selesai).
package = MODEL_DIR / 'web_deployment_package.zip'
with zipfile.ZipFile(package, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(MODEL_DIR / 'phishing_url_bundle.joblib', 'phishing_url_bundle.joblib')
    zf.write(MODEL_DIR / 'feature_extraction.py', 'feature_extraction.py')
    zf.write('app_flask.py', 'app_flask.py')
    zf.write('requirements.txt', 'requirements.txt')
print('Paket deployment:', package.resolve())

## **8. Catatan untuk BAB III dan BAB IV**

- **Master 98 kolom:** seluruh fitur asli PhiUSIIL (`phi_*`), dataset baru (`baru_*`), DNS (`dns_*`), dan 15 fitur standar (`model_*`) **tetap disimpan**. Nilai kosong berarti fitur tidak tersedia dari sumber tersebut.
- **Model memakai 15 fitur standar:** dipilih karena konsisten untuk **semua URL** dan bisa dihitung dari input teks URL saat website diakses.
- **Cleaning:** normalisasi URL, penghapusan duplikat, penyisihan URL dengan label bertentangan; simpan jumlahnya dari output.
- **Pembagian data:** berbasis registered domain, dipisah menjadi train, validation, dan test. Tuning juga tidak boleh melibatkan set test.
- **Evaluasi:** laporkan hasil **yang dijalankan ulang** dari notebook ini, terutama *Phishing Recall*, *Phishing F1*, dan hasil **URL tanpa `www`**.
- **Batasan:** tanpa HTML/DNS real-time, model ini mengukur risiko dari pola URL; output “Legitimate” **bukan jaminan aman**. Dataset dengan sumber/tahun berbeda juga bisa menimbulkan bias.
- **Keamanan:** file Joblib adalah pickle; **hanya load model yang dibuat dan dipercaya sendiri**.